# Wipro India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** careers.wipro.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-01 01:06:30
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Wipro"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Wipro/Outputs/2026_04_01


In [4]:
print("=" * 60)
print("WIPRO INDIA JOB SCRAPER")
print("Source: careers.wipro.com (Radancy/Jobs2Web platform)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


wipro_jobs = []

# Wipro uses Radancy/Jobs2Web — URL pattern: /job/TITLE/JOBID-en_US/
# Strategy 1: Try the search results page directly
driver = setup_selenium()
try:
    # Use the search page with India location
    driver.get("https://careers.wipro.com/search/?q=&location=India")
    time.sleep(10)

    # Wait for job listings to render
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*=\'/job/\']"))
        )
    except:
        print("  Waiting longer...")
        time.sleep(8)
        # Try alternate URL
        driver.get("https://careers.wipro.com/viewalljobs/")
        time.sleep(10)

    for page in range(15):
        soup = BeautifulSoup(driver.page_source, "lxml")

        # Remove nav/header/footer
        for unwanted in soup.select("nav, header, footer, [role=\'navigation\'], [class*=\'nav-\'], [class*=\'menu\']"):
            unwanted.decompose()

        # Primary selector: links to /job/ pages (Radancy pattern)
        job_links = soup.select("a[href*=\'/job/\'][href*=\'-en_US\']")
        if not job_links:
            job_links = soup.select("a[href*=\'/job/\']")

        new_count = 0
        for link in job_links:
            title = link.get_text(strip=True)
            href = link.get("href", "")

            # Extract job ID from URL pattern: /job/TITLE/JOBID-en_US/
            job_id_match = re.search(r"/job/[^/]+/(\d+)", href)
            job_id = job_id_match.group(1) if job_id_match else href.split("/")[-2] if "/" in href else str(len(wipro_jobs))

            # Get parent card for location
            card = link.parent
            loc = "India"
            if card:
                loc_el = card.select_one("[class*=\'location\'], [class*=\'city\'], span[class*=\'loc\']")
                if loc_el:
                    loc = loc_el.get_text(strip=True)

            if is_valid_job_title(title) and title not in [j["title"] for j in wipro_jobs]:
                full_url = href if href.startswith("http") else f"https://careers.wipro.com{href}" if href else ""
                wipro_jobs.append({
                    "job_id": str(job_id),
                    "title": title,
                    "company_name": "Wipro",
                    "raw_jd_text": "",
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "IT Services",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": full_url,
                    "business_unit": "",
                    "source_platform": "Wipro Radancy",
                })
                new_count += 1

        print(f"  Page {page+1}: {new_count} new jobs (total: {len(wipro_jobs)})")
        if new_count == 0 and page > 0:
            break

        # Try pagination
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label*=\'Next\'], a[aria-label*=\'next\'], [class*=\'next\'] a, a.next-btn")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(4)
        except:
            break

    # Fetch JDs for found jobs
    if wipro_jobs:
        print(f"\n  Fetching JD details for up to 40 jobs...")
        for i, job in enumerate(wipro_jobs[:40]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 100:
                continue
            if job["job_url"]:
                jd = fetch_jd_selenium(driver, job["job_url"])
                if jd:
                    wipro_jobs[i]["raw_jd_text"] = jd
            if (i + 1) % 10 == 0:
                print(f"    Fetched {i+1}/{min(40, len(wipro_jobs))} JDs")

except Exception as e:
    print(f"  Error: {e}")
    import traceback; traceback.print_exc()
finally:
    driver.quit()

print(f"Total Wipro India jobs: {len(wipro_jobs)}")


WIPRO INDIA JOB SCRAPER
Source: careers.wipro.com (Radancy/Jobs2Web platform)


  Page 1: 9 new jobs (total: 9)

  Fetching JD details for up to 40 jobs...


Total Wipro India jobs: 9


In [5]:
df_wipro = save_results(wipro_jobs, "Wipro", OUTPUT_DIR)
if df_wipro is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_wipro.columns]
    print(df_wipro[cols].head(10).to_string())


  [OK] Saved 9 jobs -> Wipro_jobs_2026-04-01.csv
       Seniority: {'junior': 6, 'lead': 3}
       Work mode: {'onsite': 9}
       Has JD text: 9/9
       Has job URL: 9/9
       Has business unit: 0/9

Sample jobs:
                                                   title location_city seniority_level business_unit                                                                                                 job_url
0                             ADMINISTRATOR L1(CONTRACT)         India            lead                                             https://careers.wipro.com/job/ADMINISTRATOR-L1%28CONTRACT%29/130553-en_US
1         Sap ABAP+EAM+MRS+Mobile Application Consultant         India          junior                       https://careers.wipro.com/job/Sap-ABAP%2BEAM%2BMRS%2BMobile-Application-Consultant/130410-en_US
2  Sales Executive (Hunter), Network Equipment Providers         India            lead                https://careers.wipro.com/job/Sales-Executive-%28Hunter%29%2C-Netwo